# Notebook 05 — Earth Engine Concepts & Pre-extracted Data

**GeoAI for Climate, Environment and Health — Zimbabwe DHS 2015 Workshop**  
25 June 2026 · YSN Bulawayo

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Identify which satellite and reanalysis products are most relevant for health-related environmental exposures.
2. Describe the conceptual GEE workflow for extracting point-level values from image collections at DHS cluster coordinates.
3. Perform temporal aggregation and compute descriptive statistics on the pre-extracted monthly climate data.
4. Apply quality flagging and gap-filling to simulated missing satellite observations.

---

**Context.** Google Earth Engine (GEE) allows extraction of satellite-derived environmental variables at specific coordinates over specified date ranges. In a real workflow, you would upload DHS cluster coordinates as a `FeatureCollection`, join it to image collections (e.g., MODIS MOD11A2 for LST), and extract mean values over temporal windows aligned to each pregnancy. For this workshop we work with pre-extracted data that mirrors exactly what a GEE workflow would produce for the 399 Zimbabwe DHS 2015 clusters over 60 months (2011–2015).

---
## Section 1 — Satellite Products for Environmental Health

A reference map from satellite product to health-relevant exposure:

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = pathlib.Path().resolve().parent / 'data'

product_table = pd.DataFrame([
    {
        'GEE Product'     : 'MODIS MOD11A2',
        'Variable'        : 'LST day (°C)',
        'Native resolution': '1 km, 8-day',
        'Health relevance': 'Heat stress, urban heat island, thermal comfort'
    },
    {
        'GEE Product'     : 'CHIRPS',
        'Variable'        : 'Rainfall (mm/day)',
        'Native resolution': '0.05°, daily',
        'Health relevance': 'Flood risk, drought, agriculture/nutrition'
    },
    {
        'GEE Product'     : 'MODIS MOD13A3',
        'Variable'        : 'NDVI (unitless)',
        'Native resolution': '1 km, monthly',
        'Health relevance': 'Vegetation cover, food production, vector habitat'
    },
    {
        'GEE Product'     : 'ERA5 / ERA5-Land',
        'Variable'        : 'Temp, wind, humidity',
        'Native resolution': '0.25°, hourly',
        'Health relevance': 'Heat index, respiratory exposures, wildfire risk'
    },
    {
        'GEE Product'     : 'VIIRS VNP46A1',
        'Variable'        : 'Nighttime lights (nW/cm²/sr)',
        'Native resolution': '500 m, daily',
        'Health relevance': 'Urbanisation proxy, electricity access, light pollution'
    },
])

pd.set_option('display.max_colwidth', 60)
print("Satellite products for environmental health research:")
print(product_table.to_string(index=False))

**Conceptual GEE extraction workflow** (no live GEE connection required for this workshop):

```python
import ee
ee.Initialize()

# 1. Upload cluster coordinates as a FeatureCollection (via Assets → New → Table upload)
clusters_fc = ee.FeatureCollection('users/your_username/zw_dhs_clusters')

# 2. Load the MODIS LST image collection, filter to date range
modis_lst = (
    ee.ImageCollection('MODIS/061/MOD11A2')
    .filterDate('2011-01-01', '2015-12-31')
    .select('LST_Day_1km')
    .map(lambda img: img.multiply(0.02).subtract(273.15))  # scale + K→°C
)

# 3. For each image, extract value at each cluster centroid
def sample_image(img):
    return img.reduceRegions(
        collection=clusters_fc,
        reducer=ee.Reducer.mean(),
        scale=1000
    ).map(lambda f: f.set('date', img.date().format('YYYY-MM')))

sampled = modis_lst.map(sample_image).flatten()

# 4. Export to Google Drive
ee.batch.Export.table.toDrive(
    collection=sampled,
    description='zw_modis_lst_2011_2015',
    fileFormat='CSV'
).start()
```

The `zw_climate_monthly.csv` we work with below is the result of exactly this workflow, run for LST (MODIS), rainfall (CHIRPS), NDVI (MODIS), and heat index (derived from ERA5) across all 399 clusters for 60 months.

---
## Section 2 — Load and Validate Pre-extracted Data

In [ ]:
climate  = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters = pd.read_csv(DATA_DIR / 'zw_clusters.csv')

n_clusters = climate['cluster_id'].nunique()
n_months   = climate.groupby('cluster_id').size().iloc[0]
print(f"Climate data: {len(climate):,} rows  ({n_clusters} clusters × {n_months} months)")
print(f"Year range: {climate['year'].min()}–{climate['year'].max()}")
print()

# Check for missing values
print("Missing values per column:")
print(climate.isnull().sum())
print()

# QC: rainfall must be non-negative
neg_rain = (climate['chirps_mm'] < 0).sum()
print(f"Negative chirps_mm values: {neg_rain}  (expected: 0)")
if neg_rain > 0:
    print("  WARNING: negative rainfall detected — clip to 0 before analysis.")
else:
    print("  QC passed: all rainfall values are non-negative.")
print()

print("Summary statistics:")
print(climate[['lst_day_c', 'chirps_mm', 'ndvi', 'heat_index']].describe().round(2))

---
## Section 3 — Exercise 1 (3 pts): Urban vs Rural LST Statistics

Merge `zw_climate_monthly.csv` with `zw_clusters.csv` on `cluster_id` to obtain the `urban` flag (1 = urban, 0 = rural) for each observation. Compute the mean, standard deviation, min, and max of `lst_day_c` separately for urban and rural clusters.

**Store:**
- `lst_urban_stats` — dict or Series with keys `'mean'`, `'std'`, `'min'`, `'max'` (values rounded to 2 dp)
- `lst_rural_stats` — same structure

The difference in means is an estimate of the urban heat island effect.

In [ ]:
### BEGIN SOLUTION

climate_urban = climate.merge(
    clusters[['cluster_id', 'urban']],
    on='cluster_id',
    how='left'
)

def lst_stats(df, urban_flag):
    sub = df.loc[df['urban'] == urban_flag, 'lst_day_c']
    return {
        'mean': round(float(sub.mean()), 2),
        'std' : round(float(sub.std()),  2),
        'min' : round(float(sub.min()),  2),
        'max' : round(float(sub.max()),  2),
    }

lst_urban_stats = lst_stats(climate_urban, urban_flag=1)
lst_rural_stats = lst_stats(climate_urban, urban_flag=0)

uhi = round(lst_urban_stats['mean'] - lst_rural_stats['mean'], 2)

print("LST statistics — urban clusters:")
for k, v in lst_urban_stats.items():
    print(f"  {k:4s}: {v:.2f} °C")
print()
print("LST statistics — rural clusters:")
for k, v in lst_rural_stats.items():
    print(f"  {k:4s}: {v:.2f} °C")
print()
print(f"Urban heat island signal (mean urban − mean rural): {uhi:+.2f} °C")

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
for stat_dict, label in [(lst_urban_stats, 'lst_urban_stats'), (lst_rural_stats, 'lst_rural_stats')]:
    assert isinstance(stat_dict, (dict, pd.Series)), f"{label} must be a dict or Series"
    for k in ('mean', 'std', 'min', 'max'):
        assert k in stat_dict, f"Key '{k}' missing from {label}"
        assert 10.0 <= float(stat_dict[k]) <= 45.0 or k == 'std', \
            f"{label}['{k}'] = {stat_dict[k]} is outside plausible LST range"
assert float(lst_urban_stats['min']) < float(lst_urban_stats['max']), \
    "Urban LST min must be less than max"
assert float(lst_rural_stats['min']) < float(lst_rural_stats['max']), \
    "Rural LST min must be less than max"
### END HIDDEN TESTS

---
## Section 4 — Worked Example: Temporal Aggregation

We demonstrate two common aggregation patterns:
1. Annual mean per cluster — for cross-sectional exposure assignment.
2. Rolling 3-month mean for a single cluster — showing how trimester windows are conceptually constructed.

In [ ]:
# --- Pattern 1: Annual mean per cluster ---
annual_mean = (
    climate
    .groupby(['cluster_id', 'year'])[['lst_day_c', 'chirps_mm', 'ndvi']]
    .mean()
    .round(3)
    .reset_index()
)
print("Annual mean climate per cluster (first 10 rows):")
print(annual_mean.head(10).to_string(index=False))

In [ ]:
# --- Pattern 2: Rolling 3-month mean for cluster 1 ---
cl1 = climate[climate['cluster_id'] == 1].sort_values(['year', 'month']).copy()
cl1['date_idx'] = range(len(cl1))

# 3-month rolling mean (min_periods=3 to avoid edge artefacts)
cl1['lst_roll3']  = cl1['lst_day_c'].rolling(3, min_periods=3).mean()
cl1['rain_roll3'] = cl1['chirps_mm'].rolling(3, min_periods=3).mean()

month_labels = [f"{y}-{m:02d}" for y, m in zip(cl1['year'], cl1['month'])]
tick_pos = list(range(0, len(cl1), 6))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(cl1['date_idx'], cl1['lst_day_c'], alpha=0.5, lw=1, color='steelblue', label='Monthly')
axes[0].plot(cl1['date_idx'], cl1['lst_roll3'], lw=2, color='firebrick', label='3-month rolling mean')
axes[0].set_ylabel('LST (°C)', fontsize=10)
axes[0].set_title('Cluster 1 — monthly vs rolling 3-month mean LST', fontsize=11)
axes[0].legend(fontsize=9)

axes[1].bar(cl1['date_idx'], cl1['chirps_mm'], alpha=0.45, color='royalblue', label='Monthly')
axes[1].plot(cl1['date_idx'], cl1['rain_roll3'], lw=2, color='darkorange', label='3-month rolling mean')
axes[1].set_ylabel('Rainfall (mm)', fontsize=10)
axes[1].set_title('Cluster 1 — monthly vs rolling 3-month mean rainfall', fontsize=11)
axes[1].legend(fontsize=9)

for ax in axes:
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([month_labels[i] for i in tick_pos], rotation=35, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

print("Note: each 3-month window is one trimester's exposure. In the DHS analysis")
print("these windows are aligned to each pregnancy's conception date, not to calendar quarters.")

---
## Section 5 — Exercise 2 (3 pts): Spatial Variability of Rainfall by Calendar Month

For each calendar month (1–12), compute the coefficient of variation (CV = std / mean × 100) of `chirps_mm` across all 399 clusters. This captures how spatially heterogeneous rainfall is within Zimbabwe in each month — dry-season months near zero should show very high CV.

**Store:**
- `rainfall_cv` — pandas Series, index = month 1–12, values = CV (%, rounded to 1 dp)
- `most_variable_rainfall_month` — int (month number with highest CV)

In [ ]:
### BEGIN SOLUTION

def cv_pct(x):
    m = x.mean()
    return (x.std() / m * 100) if m > 0 else np.nan

rainfall_cv = (
    climate
    .groupby('month')['chirps_mm']
    .apply(cv_pct)
    .round(1)
)

most_variable_rainfall_month = int(rainfall_cv.idxmax())

print("Coefficient of variation (%) of CHIRPS rainfall across 399 clusters, by month:")
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for m, cv in rainfall_cv.items():
    bar = '█' * int(cv / 5)
    print(f"  Month {m:2d} ({month_names[m-1]}): {cv:6.1f}%  {bar}")
print()
print(f"Month with highest spatial CV: {most_variable_rainfall_month} "
      f"({month_names[most_variable_rainfall_month-1]})")
print("Interpretation: in dry-season months, nearly all clusters have near-zero rainfall")
print("but a few may receive isolated convective events, producing extreme CV values.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rainfall_cv.index, rainfall_cv.values, color='royalblue', edgecolor='white')
ax.axhline(100, color='firebrick', lw=1.2, linestyle='--', label='CV = 100%')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_xlabel('Calendar month', fontsize=11)
ax.set_ylabel('CV of rainfall across clusters (%)', fontsize=11)
ax.set_title('Spatial variability (CV) of CHIRPS rainfall by month\nZimbabwe DHS clusters 2011–2015', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert isinstance(rainfall_cv, pd.Series),                   "rainfall_cv must be a pandas Series"
assert len(rainfall_cv) == 12,                                "rainfall_cv must have 12 entries (one per month)"
assert set(rainfall_cv.index) == set(range(1, 13)),           "rainfall_cv index must be months 1–12"
assert isinstance(most_variable_rainfall_month, int),          "most_variable_rainfall_month must be an int"
assert 1 <= most_variable_rainfall_month <= 12,               "most_variable_rainfall_month must be 1–12"
assert rainfall_cv[most_variable_rainfall_month] == rainfall_cv.max(), \
    "most_variable_rainfall_month must correspond to the highest CV"
### END HIDDEN TESTS

---
## Section 6 — Worked Example: Quality Flagging and Gap-Filling

In real MODIS data, cloud cover produces missing values (flagged as fill values). We simulate this by randomly setting 5% of `lst_day_c` to NaN, then fill gaps using each cluster's own seasonal mean (month-level mean across all non-missing years).

In [ ]:
np.random.seed(99)

climate_sim = climate.copy()

# Randomly flag 5% of rows as cloud-contaminated
n_missing  = int(len(climate_sim) * 0.05)
miss_idx   = np.random.choice(climate_sim.index, size=n_missing, replace=False)
true_vals  = climate_sim.loc[miss_idx, 'lst_day_c'].copy()  # store for RMSE check
climate_sim.loc[miss_idx, 'lst_day_c'] = np.nan

print(f"Simulated missing values: {climate_sim['lst_day_c'].isnull().sum()} ({5:.0f}% of {len(climate_sim):,} rows)")

# Compute seasonal mean per (cluster_id, month) from non-missing rows
seasonal_mean = (
    climate_sim
    .dropna(subset=['lst_day_c'])
    .groupby(['cluster_id', 'month'])['lst_day_c']
    .mean()
    .reset_index()
    .rename(columns={'lst_day_c': 'lst_seasonal_mean'})
)

climate_filled = climate_sim.merge(seasonal_mean, on=['cluster_id', 'month'], how='left')

# Fill gaps
still_missing = climate_filled['lst_day_c'].isnull()
climate_filled.loc[still_missing, 'lst_day_c'] = \
    climate_filled.loc[still_missing, 'lst_seasonal_mean']

# Evaluate: RMSE between filled values and true values at flagged positions
filled_at_flagged = climate_filled.loc[miss_idx, 'lst_day_c'].values
rmse = np.sqrt(np.mean((filled_at_flagged - true_vals.values) ** 2))

print(f"RMSE of gap-filled vs true values at flagged positions: {rmse:.3f} °C")
within_1c = (np.abs(filled_at_flagged - true_vals.values) <= 1.0).mean() * 100
print(f"Fraction of filled values within ±1 °C of truth: {within_1c:.1f}%")
print()
print("Conclusion: seasonal-mean imputation keeps most filled values within ±1 °C of the true LST,")
print("which is acceptable for trimester-level exposure assignment. Larger cloud gaps")
print("(e.g., rainy-season months) will show higher RMSE and require more sophisticated interpolation.")

---
## Section 7 — Exercise 3 (2 pts): Province-Level Heat Index — Wet vs Dry Season

Using the original (unflagged) `climate` data, compute the mean `heat_index` for each province in:
- **Wet season:** months 11, 12, 1, 2, 3
- **Dry season:** months 5, 6, 7, 8, 9

You will need to merge climate with clusters to get the province for each cluster.

**Store:**
- `heat_wet` — dict mapping province name → mean heat index (rounded to 2 dp)
- `heat_dry` — same structure

Which province shows the largest wet–dry contrast?

In [ ]:
### BEGIN SOLUTION

WET_MONTHS = [11, 12, 1, 2, 3]
DRY_MONTHS = [5, 6, 7, 8, 9]

climate_prov = climate.merge(
    clusters[['cluster_id', 'province']],
    on='cluster_id',
    how='left'
)

wet_df = climate_prov[climate_prov['month'].isin(WET_MONTHS)]
dry_df = climate_prov[climate_prov['month'].isin(DRY_MONTHS)]

heat_wet = (
    wet_df.groupby('province')['heat_index']
    .mean()
    .round(2)
    .to_dict()
)

heat_dry = (
    dry_df.groupby('province')['heat_index']
    .mean()
    .round(2)
    .to_dict()
)

contrast = {
    prov: round(heat_wet[prov] - heat_dry.get(prov, np.nan), 2)
    for prov in heat_wet
}

max_contrast_prov = max(contrast, key=lambda p: abs(contrast[p]))

print("Province-level mean heat index (°C):")
print(f"  {'Province':<30s}  {'Wet season':>10s}  {'Dry season':>10s}  {'Contrast':>10s}")
print("  " + "-" * 66)
for prov in sorted(heat_wet):
    w = heat_wet[prov]
    d = heat_dry.get(prov, float('nan'))
    c = contrast.get(prov, float('nan'))
    print(f"  {prov:<30s}  {w:10.2f}  {d:10.2f}  {c:+10.2f}")
print()
print(f"Province with largest wet–dry contrast: {max_contrast_prov}  ({contrast[max_contrast_prov]:+.2f} °C)")

### END SOLUTION

In [ ]:
### BEGIN HIDDEN TESTS
assert isinstance(heat_wet, dict), "heat_wet must be a dict"
assert isinstance(heat_dry, dict), "heat_dry must be a dict"
assert len(heat_wet) >= 5,         "heat_wet must contain at least 5 provinces"
assert len(heat_dry) >= 5,         "heat_dry must contain at least 5 provinces"
assert set(heat_wet.keys()) == set(heat_dry.keys()), \
    "heat_wet and heat_dry must cover the same set of provinces"
assert all(15.0 <= v <= 45.0 for v in heat_wet.values()), \
    "Wet season heat index values out of plausible range (15–45 °C)"
assert all(15.0 <= v <= 45.0 for v in heat_dry.values()), \
    "Dry season heat index values out of plausible range (15–45 °C)"
### END HIDDEN TESTS

---
## Recap

| Concept | Key point |
|---|---|
| Satellite products | MODIS LST → heat; CHIRPS → rainfall; MODIS NDVI → vegetation; ERA5 → heat index |
| GEE workflow | Upload FeatureCollection → join to ImageCollection → `reduceRegions` → export to Drive |
| Urban heat island | Urban LST systematically higher than rural; detectable even at 1 km MODIS resolution |
| Spatial rainfall CV | Highest in dry-season months when most clusters record near-zero but a few receive convective events |
| Quality flagging | Seasonal-mean imputation recovers most cloud-flagged LST values within ±1 °C |
| Wet/dry contrast | Low-altitude provinces (e.g., Matabeleland) show the largest seasonal heat index contrast |

**Next:** Notebook 06 integrates all prior work — spatial joins, trimester exposure windows, and satellite data — into a full exposure characterisation pipeline and introduces the GeoAI interface for exploratory analysis.